In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_online_retail.csv")

df.columns = df.columns.str.strip()
df.rename(columns={"CustomerID": "Customer ID"}, inplace=True)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [3]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
snapshot_date = df["InvoiceDate"].max()
snapshot_date

Timestamp('2011-12-09 12:50:00')

In [4]:
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice": "nunique",
    "Revenue": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,325,12,77556.46
12347.0,1,8,4921.53
12348.0,74,5,2019.40
12349.0,18,4,4428.69
12350.0,309,1,334.40


In [5]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4,3,2,1])
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1,2,3,4])
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 4, labels=[1,2,3,4])

In [6]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,,,,
12346.0,325,12,77556.46,2,4,4,244
12347.0,1,8,4921.53,4,4,4,444
12348.0,74,5,2019.40,3,3,3,333
12349.0,18,4,4428.69,4,3,4,434
12350.0,309,1,334.40,2,1,1,211


In [ ]:
rfm.to_csv("../data/processed/rfm_features.csv", index=False)

In [2]:
import os
print(os.getcwd())

/Users/khushisingh/Desktop/ecommerce-analytics-ml/notebooks


In [2]:
import pandas as pd

df = pd.read_csv("../data/cleaned/cleaned_online_retail.csv")

df.columns = df.columns.str.strip()
df.rename(columns={"CustomerID": "Customer ID"}, inplace=True)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [3]:
snapshot_date = df["InvoiceDate"].max()

In [4]:
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice": "nunique",
    "Revenue": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,325,12,77556.46
12347.0,1,8,4921.53
12348.0,74,5,2019.40
12349.0,18,4,4428.69
12350.0,309,1,334.40


In [5]:
import os

os.makedirs("../data/processed", exist_ok=True)

rfm.to_csv("../data/processed/rfm_features.csv", index=False)

print("RFM file saved successfully")

RFM file saved successfully
